# 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from skl2onnx import convert_sklearn, update_registered_converter
from onnxmltools.convert import convert_lightgbm, convert_xgboost
from skl2onnx.common.data_types import FloatTensorType, Int64TensorType
import onnxruntime as rt
import warnings
warnings.filterwarnings('ignore')


C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-ml.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-operators-ml.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-data.proto. Please update the gencode to avoid compatibility violations in the next ru

# 2. Data Loading & Categorical Encoding

In [2]:
# 1. Data Loading from CSV
print("Loading data...")
df = pd.read_csv('electronics_history.csv')

# Convert timestamp and extract month block
df['scrape_timestamp'] = pd.to_datetime(df['scrape_timestamp'])
df['year_month'] = df['scrape_timestamp'].dt.to_period('M')

# Define features
target = 'volatility_score'
categorical_features = ['cpu_tier', 'gpu_tier']
numerical_features = [
    'base_price_egp', 'official_egp_usd', 'parallel_egp_usd', 'cpi_inflation',
    'is_major_sale_period', 'price_egp', 'price_diff', 'price_lag_7d', 'price_lag_14d',
    'price_momentum_7d', 'competitor_scarcity_count', 'ram_gb', 'storage_capacity_gb'
]
features = numerical_features + categorical_features

# 2. Categorical Encoding for ONNX compatibility
encoder = OrdinalEncoder(encoded_missing_value=np.nan)
df[categorical_features] = encoder.fit_transform(df[categorical_features])

# Sort strictly by product and time to prevent look-ahead
df = df.sort_values(by=['product_id', 'scrape_timestamp']).reset_index(drop=True)

print("Data Shape:", df.shape)
print("Time range:", df['scrape_timestamp'].min(), "to", df['scrape_timestamp'].max())


Loading data...
Data Shape: (678410, 20)
Time range: 2025-04-16 00:00:00 to 2026-04-29 00:00:00


# 3. Time-Series Split Evaluator

In [3]:
# 3. Rolling Time-Series Split (Forward Chaining)
def evaluate_model_cv(df, model_class, params, fit_kwargs=None):
    """
    Evaluates a model using rolling time-series validation.
    Trains on months < T, validates on month T.
    """
    if fit_kwargs is None: fit_kwargs = {}
    
    months = sorted(df['year_month'].unique())
    val_losses_mae = []
    
    print(f"--- Evaluating {model_class.__name__} ---")
    
    # Require at least 2 months to start training -> validate
    for val_month_idx in range(2, len(months)):
        train_months = months[:val_month_idx]
        val_month = months[val_month_idx]
        
        train_idx = df[df['year_month'].isin(train_months)].index
        val_idx = df[df['year_month'] == val_month].index
        
        X_train, y_train = df.loc[train_idx, features], df.loc[train_idx, target]
        X_val, y_val = df.loc[val_idx, features], df.loc[val_idx, target]
        
        model = model_class(**params)
        model.fit(X_train, y_train, **fit_kwargs)
        
        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        val_losses_mae.append(mae)
        print(f"Validation Month {val_month}: MAE = {mae:.4f}")
        
    avg_mae = np.mean(val_losses_mae)
    print(f"\n=> Average MAE across splits: {avg_mae:.4f}\n")
    return avg_mae


# 4. Train & Evaluate LightGBM

In [4]:
# --- Model 1: LightGBM ---
lgb_params = {
    'objective': 'huber',
    'alpha': 1.5,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'random_state': 42,
    'verbosity': -1,
    'n_estimators': 100
}

# Evaluate
evaluate_model_cv(
    df, 
    lgb.LGBMRegressor, 
    lgb_params, 
    fit_kwargs={'categorical_feature': categorical_features}
)

# Train Final on All Data
print("Training final LightGBM on all data...")
final_lgb = lgb.LGBMRegressor(**lgb_params)
final_lgb.fit(df[features], df[target], categorical_feature=categorical_features)
print("Done.")


--- Evaluating LGBMRegressor ---
Validation Month 2025-06: MAE = 17.3855
Validation Month 2025-07: MAE = 7.3128
Validation Month 2025-08: MAE = 9.7648
Validation Month 2025-09: MAE = 19.0161
Validation Month 2025-10: MAE = 5.2909
Validation Month 2025-11: MAE = 13.1363
Validation Month 2025-12: MAE = 14.7086
Validation Month 2026-01: MAE = 5.2672
Validation Month 2026-02: MAE = 4.9199
Validation Month 2026-03: MAE = 15.6482
Validation Month 2026-04: MAE = 8.1877

=> Average MAE across splits: 10.9671

Training final LightGBM on all data...
Done.


# 5. Train & Evaluate XGBoost

In [5]:
# --- Model 2: XGBoost ---
xgb_params = {
    'objective': 'reg:pseudohubererror',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 100,
    'random_state': 42
}

# Evaluate
evaluate_model_cv(df, xgb.XGBRegressor, xgb_params)

# Train Final on All Data
print("Training final XGBoost on all data...")
final_xgb = xgb.XGBRegressor(**xgb_params)
final_xgb.fit(df[features], df[target])
print("Done.")


--- Evaluating XGBRegressor ---
Validation Month 2025-06: MAE = 50.3132
Validation Month 2025-07: MAE = 103.0133
Validation Month 2025-08: MAE = 2628.5749
Validation Month 2025-09: MAE = 66.2809
Validation Month 2025-10: MAE = 111.8065
Validation Month 2025-11: MAE = 118.9778
Validation Month 2025-12: MAE = 136.1547
Validation Month 2026-01: MAE = 169.8218
Validation Month 2026-02: MAE = 186.6049
Validation Month 2026-03: MAE = 181.3705
Validation Month 2026-04: MAE = 217.7852

=> Average MAE across splits: 360.9731

Training final XGBoost on all data...
Done.


# 6. Train & Evaluate HistGradientBoosting

In [6]:
# --- Model 3: HistGradientBoostingRegressor (Scikit-Learn) ---
hgb_params = {
    'loss': 'absolute_error',
    'max_depth': 6,
    'learning_rate': 0.05,
    'max_iter': 100,
    'random_state': 42
}

# Evaluate
evaluate_model_cv(df, HistGradientBoostingRegressor, hgb_params)

# Train Final on All Data
print("Training final HistGradientBoosting on all data...")
final_hgb = HistGradientBoostingRegressor(**hgb_params)
final_hgb.fit(df[features], df[target])
print("Done.")


--- Evaluating HistGradientBoostingRegressor ---
Validation Month 2025-06: MAE = 15.7792
Validation Month 2025-07: MAE = 8.5617
Validation Month 2025-08: MAE = 13.8813
Validation Month 2025-09: MAE = 14.0437
Validation Month 2025-10: MAE = 3.7525
Validation Month 2025-11: MAE = 11.5482
Validation Month 2025-12: MAE = 13.2405
Validation Month 2026-01: MAE = 3.5261
Validation Month 2026-02: MAE = 3.6693
Validation Month 2026-03: MAE = 10.0688
Validation Month 2026-04: MAE = 7.1807

=> Average MAE across splits: 9.5684

Training final HistGradientBoosting on all data...
Done.


# 7. Final Model Selection & ONNX Export

In [7]:
# --- Final Selection & ONNX Export ---
# CHOOSE YOUR FINAL MODEL HERE based on the outputs above:
# Options: 'LightGBM', 'XGBoost', 'HistGradientBoosting'

CHOICE = 'HistGradientBoosting'  # Change this string to select

if CHOICE == 'LightGBM':
    selected_model = final_lgb
    convert_func = convert_lightgbm
elif CHOICE == 'XGBoost':
    selected_model = final_xgb
    convert_func = convert_xgboost
elif CHOICE == 'HistGradientBoosting':
    selected_model = final_hgb
    convert_func = convert_sklearn

print(f"Exporting {CHOICE} to ONNX...")

# Define Types
initial_types = [('float_input', FloatTensorType([None, len(features)]))]

# Convert
onnx_model = convert_func(selected_model, initial_types=initial_types, target_opset=12)

# Save
onnx_filename = "volatility_model.onnx"
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model serialized to {onnx_filename}")

# Validation of ONNX Inference
sess = rt.InferenceSession(onnx_filename)
input_name = sess.get_inputs()[0].name
dummy_input = df[features].iloc[[0]].values.astype(np.float32)
pred_onnx = sess.run(None, {input_name: dummy_input})[0]

print(f"ONNX Dummy Prediction ({CHOICE}):", pred_onnx)


Exporting HistGradientBoosting to ONNX...
Model serialized to volatility_model.onnx
ONNX Dummy Prediction (HistGradientBoosting): [[0.2511196]]
